# 07 External Boosting Model Tuning

Goal: compare LightGBM, XGBoost, and CatBoost on the current best feature set, then tune the strongest model. The holdout test set is evaluated only after selecting the best tuned model and threshold from validation data.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.pipeline import Pipeline

from src.features import BASIC_APPLICATION_FEATURES, create_features
from src.preprocessing import build_preprocessor
from src.thresholding import (
    evaluate_probabilities,
    find_best_threshold,
    get_positive_probabilities,
)


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2
N_TUNING_ITERATIONS = 8

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(
    RAW_DATA_DIR / "application_train.csv"
)

y = train_df["TARGET"].copy()

X = train_df[BASIC_APPLICATION_FEATURES].copy()
X = create_features(X)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()
cat_feature_indices = [
    X_train.columns.get_loc(column)
    for column in categorical_columns
]


def make_catboost_safe_frame(data):
    data = data.copy()
    for column in categorical_columns:
        data[column] = data[column].astype("object").where(
            data[column].notna(),
            "__MISSING__",
        )
        data[column] = data[column].astype(str)
    return data


X_train_cat = make_catboost_safe_frame(X_train)
X_valid_cat = make_catboost_safe_frame(X_valid)
X_train_full_cat = make_catboost_safe_frame(X_train_full)
X_test_cat = make_catboost_safe_frame(X_test)

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count
scale_pos_weight = negative_count / positive_count

{
    "feature_shape": X.shape,
    "train_shape": X_train.shape,
    "valid_shape": X_valid.shape,
    "test_shape": X_test.shape,
    "categorical_columns": categorical_columns,
    "scale_pos_weight": scale_pos_weight,
}


{'feature_shape': (307511, 32),
 'train_shape': (196806, 32),
 'valid_shape': (49202, 32),
 'test_shape': (61503, 32),
 'categorical_columns': ['CODE_GENDER',
  'NAME_EDUCATION_TYPE',
  'NAME_FAMILY_STATUS',
  'NAME_INCOME_TYPE',
  'NAME_HOUSING_TYPE',
  'OCCUPATION_TYPE'],
 'scale_pos_weight': np.float64(11.387084592145015)}

In [3]:
def make_encoded_pipeline(model):
    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        one_hot_sparse_output=True,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def evaluate_model_candidate(model_name, model, threshold=None):
    if model_name.startswith("CatBoost"):
        model.fit(
            X_train_cat,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid_cat, y_valid),
            verbose=False,
        )
        valid_probabilities = model.predict_proba(X_valid_cat)[:, 1]
    else:
        pipeline = make_encoded_pipeline(model)
        pipeline.fit(X_train, y_train)
        valid_probabilities = get_positive_probabilities(pipeline, X_valid)
        model = pipeline

    threshold_result = find_best_threshold(
        y_true=y_valid,
        probabilities=valid_probabilities,
    )

    selected_threshold = threshold_result["threshold"] if threshold is None else threshold

    metrics = evaluate_probabilities(
        y_true=y_valid,
        probabilities=valid_probabilities,
        threshold=selected_threshold,
    )

    result = {
        "model": model_name,
        "threshold": selected_threshold,
        "validation_roc_auc": metrics["roc_auc"],
        "validation_average_precision": metrics["average_precision"],
        "validation_precision_class_1": metrics["precision_class_1"],
        "validation_recall_class_1": metrics["recall_class_1"],
        "validation_f1_class_1": metrics["f1_class_1"],
    }

    return result, model


In [4]:
baseline_candidates = {
    "LightGBM baseline": LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
    "XGBoost baseline": XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "CatBoost baseline": CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
    ),
}


In [5]:
baseline_results = []
fitted_baseline_models = {}

for model_name, model in baseline_candidates.items():
    print(f"Fitting: {model_name}")
    result, fitted_model = evaluate_model_candidate(model_name, model)
    baseline_results.append(result)
    fitted_baseline_models[model_name] = fitted_model

baseline_df = pd.DataFrame(baseline_results).sort_values(
    ["validation_roc_auc", "validation_f1_class_1"],
    ascending=False,
).reset_index(drop=True)

baseline_df


Fitting: LightGBM baseline


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitting: XGBoost baseline


Fitting: CatBoost baseline


,model,threshold,validation_roc_auc,validation_average_precision,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1
0,LightGBM baseline,0.680605,0.757775,0.238682,0.250000,0.393001,0.305599
1,CatBoost baseline,0.668477,0.756222,0.235139,0.238993,0.418177,0.304157
2,XGBoost baseline,0.666839,0.754955,0.234712,0.235236,0.418177,0.301097


In [6]:
best_family = baseline_df.loc[0, "model"].replace(" baseline", "")
best_family


'LightGBM'

In [7]:
if best_family == "LightGBM":
    tuning_space = {
        "n_estimators": [400, 600, 800],
        "learning_rate": [0.02, 0.03, 0.05],
        "num_leaves": [15, 31, 63],
        "min_child_samples": [20, 50, 100],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "reg_lambda": [0.0, 1.0, 5.0, 10.0],
    }

    def make_tuned_model(params):
        return LGBMClassifier(
            **params,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )

elif best_family == "XGBoost":
    tuning_space = {
        "n_estimators": [400, 600, 800],
        "learning_rate": [0.02, 0.03, 0.05],
        "max_depth": [2, 3, 4],
        "min_child_weight": [3, 5, 10],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "reg_lambda": [1.0, 5.0, 10.0],
    }

    def make_tuned_model(params):
        return XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

else:
    tuning_space = {
        "iterations": [400, 600, 800],
        "learning_rate": [0.02, 0.03, 0.05],
        "depth": [4, 5, 6, 7],
        "l2_leaf_reg": [3, 5, 10, 20],
        "bagging_temperature": [0.0, 0.5, 1.0],
    }

    def make_tuned_model(params):
        return CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            allow_writing_files=False,
            verbose=False,
        )

sampled_params = list(ParameterSampler(
    tuning_space,
    n_iter=N_TUNING_ITERATIONS,
    random_state=RANDOM_STATE,
))

sampled_params


[{'subsample': 1.0,
  'reg_lambda': 5.0,
  'num_leaves': 63,
  'n_estimators': 800,
  'min_child_samples': 50,
  'learning_rate': 0.05,
  'colsample_bytree': 0.7},
 {'subsample': 0.85,
  'reg_lambda': 10.0,
  'num_leaves': 63,
  'n_estimators': 800,
  'min_child_samples': 100,
  'learning_rate': 0.02,
  'colsample_bytree': 0.85},
 {'subsample': 1.0,
  'reg_lambda': 0.0,
  'num_leaves': 31,
  'n_estimators': 600,
  'min_child_samples': 50,
  'learning_rate': 0.02,
  'colsample_bytree': 0.85},
 {'subsample': 0.7,
  'reg_lambda': 1.0,
  'num_leaves': 31,
  'n_estimators': 400,
  'min_child_samples': 50,
  'learning_rate': 0.02,
  'colsample_bytree': 0.85},
 {'subsample': 0.7,
  'reg_lambda': 5.0,
  'num_leaves': 31,
  'n_estimators': 400,
  'min_child_samples': 20,
  'learning_rate': 0.05,
  'colsample_bytree': 0.85},
 {'subsample': 0.7,
  'reg_lambda': 10.0,
  'num_leaves': 15,
  'n_estimators': 400,
  'min_child_samples': 100,
  'learning_rate': 0.02,
  'colsample_bytree': 1.0},
 {'subs

In [8]:
tuning_results = []
fitted_tuned_models = {}

for index, params in enumerate(sampled_params, start=1):
    model_name = f"{best_family} tuned {index}"
    print(f"Fitting: {model_name}")
    print(params)

    model = make_tuned_model(params)
    result, fitted_model = evaluate_model_candidate(model_name, model)
    result["params"] = params
    tuning_results.append(result)
    fitted_tuned_models[model_name] = fitted_model

tuning_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_f1_class_1", "validation_roc_auc"],
    ascending=False,
).reset_index(drop=True)

tuning_df


Fitting: LightGBM tuned 1
{'subsample': 1.0, 'reg_lambda': 5.0, 'num_leaves': 63, 'n_estimators': 800, 'min_child_samples': 50, 'learning_rate': 0.05, 'colsample_bytree': 0.7}


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitting: LightGBM tuned 2
{'subsample': 0.85, 'reg_lambda': 10.0, 'num_leaves': 63, 'n_estimators': 800, 'min_child_samples': 100, 'learning_rate': 0.02, 'colsample_bytree': 0.85}


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitting: LightGBM tuned 3
{'subsample': 1.0, 'reg_lambda': 0.0, 'num_leaves': 31, 'n_estimators': 600, 'min_child_samples': 50, 'learning_rate': 0.02, 'colsample_bytree': 0.85}


...

Fitting: LightGBM tuned 7
{'subsample': 0.85, 'reg_lambda': 10.0, 'num_leaves': 63, 'n_estimators': 400, 'min_child_samples': 50, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fitting: LightGBM tuned 8
{'subsample': 1.0, 'reg_lambda': 0.0, 'num_leaves': 31, 'n_estimators': 600, 'min_child_samples': 100, 'learning_rate': 0.02, 'colsample_bytree': 0.85}


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,threshold,validation_roc_auc,validation_average_precision,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1,params
0,LightGBM tuned 3,0.679819,0.759463,0.241413,0.248490,0.403827,0.307663,"{'subsample': 1.0, 'reg_lambda': 0.0, 'num_lea..."
1,LightGBM tuned 7,0.651243,0.758903,0.240978,0.235928,0.441088,0.307422,"{'subsample': 0.85, 'reg_lambda': 10.0, 'num_l..."
2,LightGBM tuned 2,0.633791,0.758174,0.241084,0.230769,0.457704,0.306835,"{'subsample': 0.85, 'reg_lambda': 10.0, 'num_l..."
3,LightGBM tuned 5,0.641017,0.759918,0.241508,0.228941,0.463243,0.306437,"{'subsample': 0.7, 'reg_lambda': 5.0, 'num_lea..."
4,LightGBM tuned 8,0.638318,0.759630,0.239783,0.225918,0.475327,0.306270,"{'subsample': 1.0, 'reg_lambda': 0.0, 'num_lea..."
5,LightGBM tuned 4,0.633219,0.759007,0.238088,0.222375,0.489930,0.305903,"{'subsample': 0.7, 'reg_lambda': 1.0, 'num_lea..."
6,LightGBM tuned 1,0.608786,0.753372,0.233303,0.234708,0.432779,0.304356,"{'subsample': 1.0, 'reg_lambda': 5.0, 'num_lea..."
7,LightGBM tuned 6,0.698806,0.756501,0.236790,0.253014,0.369839,0.300470,"{'subsample': 0.7, 'reg_lambda': 10.0, 'num_le..."


In [9]:
all_model_results = pd.concat(
    [
        baseline_df.assign(stage="baseline"),
        tuning_df.assign(stage="tuned"),
    ],
    ignore_index=True,
).sort_values(
    ["validation_f1_class_1", "validation_roc_auc"],
    ascending=False,
).reset_index(drop=True)

best_model_row = all_model_results.iloc[0]
best_model_name = best_model_row["model"]
best_threshold = best_model_row["threshold"]
best_params = best_model_row.get("params", None)

all_model_results


,model,threshold,validation_roc_auc,validation_average_precision,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1,stage,params
0,LightGBM tuned 3,0.679819,0.759463,0.241413,0.248490,0.403827,0.307663,tuned,"{'subsample': 1.0, 'reg_lambda': 0.0, 'num_lea..."
1,LightGBM tuned 7,0.651243,0.758903,0.240978,0.235928,0.441088,0.307422,tuned,"{'subsample': 0.85, 'reg_lambda': 10.0, 'num_l..."
2,LightGBM tuned 2,0.633791,0.758174,0.241084,0.230769,0.457704,0.306835,tuned,"{'subsample': 0.85, 'reg_lambda': 10.0, 'num_l..."
3,LightGBM tuned 5,0.641017,0.759918,0.241508,0.228941,0.463243,0.306437,tuned,"{'subsample': 0.7, 'reg_lambda': 5.0, 'num_lea..."
4,LightGBM tuned 8,0.638318,0.759630,0.239783,0.225918,0.475327,0.306270,tuned,"{'subsample': 1.0, 'reg_lambda': 0.0, 'num_lea..."
5,LightGBM tuned 4,0.633219,0.759007,0.238088,0.222375,0.489930,0.305903,tuned,"{'subsample': 0.7, 'reg_lambda': 1.0, 'num_lea..."
6,LightGBM baseline,0.680605,0.757775,0.238682,0.250000,0.393001,0.305599,baseline,NaN
7,LightGBM tuned 1,0.608786,0.753372,0.233303,0.234708,0.432779,0.304356,tuned,"{'subsample': 1.0, 'reg_lambda': 5.0, 'num_lea..."
8,CatBoost baseline,0.668477,0.756222,0.235139,0.238993,0.418177,0.304157,baseline,NaN
9,XGBoost baseline,0.666839,0.754955,0.234712,0.235236,0.418177,0.301097,baseline,NaN


In [10]:
best_model_name, best_threshold, best_params


('LightGBM tuned 3',
 np.float64(0.6798187129352671),
 {'subsample': 1.0,
  'reg_lambda': 0.0,
  'num_leaves': 31,
  'n_estimators': 600,
  'min_child_samples': 50,
  'learning_rate': 0.02,
  'colsample_bytree': 0.85})

In [11]:
def refit_final_model(model_name, params):
    if model_name.startswith("LightGBM"):
        if not isinstance(params, dict):
            params = {
                "n_estimators": 500,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
            }

        model = LGBMClassifier(
            **params,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        pipeline = make_encoded_pipeline(model)
        pipeline.fit(X_train_full, y_train_full)
        return pipeline

    if model_name.startswith("XGBoost"):
        if not isinstance(params, dict):
            params = {
                "n_estimators": 500,
                "learning_rate": 0.03,
                "max_depth": 3,
                "min_child_weight": 5,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
            }

        full_positive_count = y_train_full.sum()
        full_negative_count = len(y_train_full) - full_positive_count
        full_scale_pos_weight = full_negative_count / full_positive_count

        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            scale_pos_weight=full_scale_pos_weight,
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        pipeline = make_encoded_pipeline(model)
        pipeline.fit(X_train_full, y_train_full)
        return pipeline

    if not isinstance(params, dict):
        params = {
            "iterations": 500,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 5,
        }

    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(
        X_train_full_cat,
        y_train_full,
        cat_features=cat_feature_indices,
        verbose=False,
    )
    return model


final_model = refit_final_model(best_model_name, best_params)

if best_model_name.startswith("CatBoost"):
    test_probabilities = final_model.predict_proba(X_test_cat)[:, 1]
else:
    test_probabilities = get_positive_probabilities(final_model, X_test)

final_default_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=0.5,
)

final_tuned_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=best_threshold,
)

final_test_df = pd.DataFrame([
    {
        "model": best_model_name,
        "threshold_strategy": "default_0.5",
        **final_default_metrics,
    },
    {
        "model": best_model_name,
        "threshold_strategy": "validation_selected",
        **final_tuned_metrics,
    },
])

final_test_df


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,LightGBM tuned 3,default_0.5,0.500000,0.764133,0.253209,0.709705,0.171952,0.680363,0.274523
1,LightGBM tuned 3,validation_selected,0.679819,0.764133,0.253209,0.857405,0.257798,0.407855,0.315913


In [12]:
from sklearn.metrics import confusion_matrix, classification_report

for threshold_strategy, threshold in [
    ("default_0.5", 0.5),
    ("validation_selected", best_threshold),
]:
    predictions = (test_probabilities >= threshold).astype(int)

    print(threshold_strategy)
    print("threshold:", threshold)
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions, zero_division=0))


default_0.5
threshold: 0.5
[[40271 16267]
 [ 1587  3378]]
              precision    recall  f1-score   support

           0       0.96      0.71      0.82     56538
           1       0.17      0.68      0.27      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.71      0.77     61503

validation_selected
threshold: 0.6798187129352671
[[50708  5830]
 [ 2940  2025]]
              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56538
           1       0.26      0.41      0.32      4965

    accuracy                           0.86     61503
   macro avg       0.60      0.65      0.62     61503
weighted avg       0.89      0.86      0.87     61503



## Interpretation Template

Use the baseline table to choose which external boosting family is strongest. Use the tuning table to select the best model configuration on validation data. Report the final holdout metrics from `final_test_df`.
